In [1]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from structural_tools.notebook import (
    check_value,  # noqa: F401
    display_table,
    display_text,
    set_params_columns,
    sig_figs,  # noqa: F401
)
from structural_tools.seismic.asce.parameters import LOAD_FACTOR_SEISMIC_ASD, RiskCategory, SeismicParameters, SiteClass

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [2]:
set_params_columns(4)

In [3]:
%%render params
U_corridor = 35 / 1000 * ksf
U_floor = 35 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [4]:
set_params_columns(5)

In [5]:
%%render params
L_floor = 230 * ft
w_floor = (30 + 9 / 12) * ft
L_corridor = 230 * ft
w_corridor = 5.5 * ft
w_building = (w_floor * 2) + w_corridor

<IPython.core.display.Latex object>

The wall heights vary across the levels.

In [6]:
%%render params
h_wall__1 = 11.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10 * ft

<IPython.core.display.Latex object>

There is a small jog in the building floorplan. This does not affect the area, but the perimeter will be longer.

In [7]:
%%render params
L_jog = 5 * ft

<IPython.core.display.Latex object>

In [8]:
%%render
P_wall = (2 * L_floor) + (2 * w_building) + (2 * L_jog)

<IPython.core.display.Latex object>

Areas can now be calculated. Note that the areas of the walls are the actual wall areas; they have not been distributed to each floor based on tributary area, yet.

In [9]:
%%render 
A_floor = 2 * (L_floor * w_floor)
# This is a slight overestimate as the corridor doesn't stretch the full length of the building
A_corridor = L_corridor * w_corridor
A_level = L_floor * w_building
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [10]:
%%render
W_floor__typ = U_floor * A_floor
W_corridor__typ = U_corridor * A_corridor

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4.

In [11]:
set_params_columns(3)

In [12]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_corridor__2 = W_corridor__typ
W_corridor__3 = W_corridor__typ
W_corridor__4 = W_corridor__typ

<IPython.core.display.Latex object>

The roof weight is

In [13]:
%%render
W_roof = U_roof * A_level

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [14]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__roof = U_wall * (A_wall__4 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [15]:
%%render 
W_level__2 = W_floor__2 + W_corridor__2 + W_wall__2
W_level__3 = W_floor__3 + W_corridor__3 + W_wall__3
W_level__4 = W_floor__4 + W_corridor__4 + W_wall__4
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [16]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [17]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [18]:
set_params_columns(5)

In [19]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [20]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [21]:
set_params_columns(1)

In [22]:
%%render params
h_top__roof = (58 + 3 / 12) * ft
h_bottom__roof = 42.5 * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [23]:
%%render
h_n__check = check_value(h_n, 65 * ft, "leq")

<IPython.core.display.Latex object>

We will approximate the building period.

In [24]:
h_n = float(h_n)  # to prevent weirdness with units

In [25]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [26]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [27]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [28]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [29]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [30]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [31]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [32]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [33]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [34]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Ground", "Level 2", "Level 3", "Level 4", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [35]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [36]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,421.490,42.5,17913.325,0.306127,72.387392,0.000000,0.000000
Level 4,10.0,613.642,32.5,19943.365,0.340819,80.590744,72.387392,723.873920
Level 3,10.5,615.454,22.0,13539.988,0.231389,54.714824,152.978136,2330.144343
Level 2,10.5,619.078,11.5,7119.397,0.121666,28.769342,207.692959,4510.920416
Ground,11.5,0.000,0.0,0.000,0.000000,0.000000,236.462302,7230.236885


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Shear Wall Design

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [37]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Roof", "Level 4", "Level 3", "Level 2"]
walls = {
    "A-2": "N-S",
    "A-2.4": "N-S",
    "A-3.7": "N-S",
    "A-4": "N-S",
    "A-5": "N-S",
    "A-5.1": "N-S",
    "A-6": "N-S",
    "A-6.1": "N-S",
    "A-6.8": "N-S",
    "A-7": "N-S",
    "A-7.8": "N-S",
    "A-8": "N-S",
    "A-8.2": "N-S",
    "A-8.9": "N-S",
    "A-9": "N-S",
    # E-W
    "A-B--A-1": "E-W",
    "A-B--A-2.5": "E-W",
    "A-B--A-3.5": "E-W",
    "A-B--A-4": "E-W",
    "A-B--A-4.8": "E-W",
    "A-B--A-5.5": "E-W",
    "A-B--A-6.5": "E-W",
    "A-B--A-7.9": "E-W",
    "A-B--A-8.8": "E-W",
    "A-C--A-1.8": "E-W",
    "A-C--A-3.2": "E-W",
    "A-C--A-3.8": "E-W",
    "A-C--A-4.6": "E-W",
    "A-C--A-5.5": "E-W",
    "A-C--A-6.5": "E-W",
    "A-C--A-6.9": "E-W",
    "A-C--A-8.5": "E-W",
    "A-C--A-9.2": "E-W",
}

index = pd.MultiIndex.from_product(
    [levels, walls.keys()],
    names=["Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (corridors are 17% heavier than the rest of the floors, for instance), but the vast majority of the area of each level is typical floor weight (30 psf) instead of corridor weight (35 psf).

$$
f_{p,x} = F_x / A_{level}
$$

In [38]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

The walls that are close to inline (A-5 and A-5.1, A-6 and A-6.1, and A-9 and A-9.1) in reality will be treated as inline and have collectors to transfer the shear load from the diaphragm to those walls.

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns} through \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Shear capacity and stiffness values come from **AWC SDPWS 2021 Table 4.3A**.

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [39]:
from structural_tools.seismic.timber.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [40]:
# Plan dimensions
l_A1__A2 = (31 + 11 / 12) * ft
l_A2__A3 = (17 + 11 / 12) * ft
l_A3__A4 = (26 + 4 / 12) * ft
l_A4__A5 = (21 + 4 / 12) * ft
l_A5__A6 = (24 + 8 / 12) * ft
l_A6__A7 = (34 + 4 / 12) * ft
l_A7__A8 = (26 + 1 / 12) * ft
l_A8__A9 = (21 + 4 / 12) * ft
l_A9__A10 = (31 + 1 / 12) * ft

l_A1__A24 = (40 + 1 / 12) * ft
l_A24__A35 = (23 + 8 / 12) * ft
l_A35__A51 = (34 + 4 / 12) * ft
l_A51__A61 = (24 + 8 / 12) * ft
l_A61__A68 = (24 + 8 / 12) * ft
l_A68__A82 = (38 + 3 / 12) * ft
l_A82__A89 = (17 + 11 / 12) * ft
l_A89__A10 = (31 + 5 / 12) * ft

l_A37__A4 = (8 + 10 / 12) * ft
l_A3__A37 = l_A3__A4 - l_A37__A4
l_A78__A82 = (8 + 10 / 12) * ft
l_A68__A78 = l_A68__A82 - l_A78__A82

d_trib_ns = 67 / 2 * ft

In [41]:
# Wall tributary widths and lengths
t_w__A2 = l_A1__A2 + (l_A2__A3 + l_A3__A37) / 2
t_w__A24 = (l_A1__A24 - L_jog) + ((l_A24__A35 + l_A35__A51) / 2)
t_w__A37 = ((l_A2__A3 + l_A3__A37) / 2) + (l_A37__A4 / 2)
t_w__A4 = (l_A37__A4 / 2) + (l_A4__A5 / 2)
t_w__A5 = (l_A4__A5 / 2) + (l_A5__A6 / 2)
t_w__A51 = ((l_A24__A35 + l_A35__A51) / 2) + (l_A51__A61 / 2)
t_w__A6 = (l_A5__A6 / 2) + (l_A6__A7 / 2)
t_w__A61 = (l_A51__A61 / 2) + (l_A61__A68 / 2)
t_w__A68 = (l_A61__A68 / 2) + (l_A68__A78 / 2)
t_w__A7 = (l_A6__A7 / 2) + (l_A7__A8 / 2)
t_w__A78 = (l_A68__A78 / 2) + (l_A78__A82 / 2)
t_w__A8 = (l_A7__A8 / 2) + (l_A8__A9 / 2)
t_w__A82 = (l_A78__A82 / 2) + (l_A82__A89 / 2)
t_w__A89 = (l_A82__A89 / 2) + l_A89__A10
t_w__A9 = (l_A8__A9 / 2) + (l_A9__A10 - L_jog)

t_l__A2 = float(d_trib_ns)
t_l__A24 = float(d_trib_ns)
t_l__A37 = float(d_trib_ns)
t_l__A4 = float(d_trib_ns)
t_l__A5 = float(d_trib_ns)
t_l__A51 = float(d_trib_ns)
t_l__A6 = float(d_trib_ns)
t_l__A61 = float(d_trib_ns)
t_l__A68 = float(d_trib_ns)
t_l__A7 = float(d_trib_ns)
t_l__A78 = float(d_trib_ns)
t_l__A8 = float(d_trib_ns)
t_l__A82 = float(d_trib_ns)
t_l__A89 = float(d_trib_ns)
t_l__A9 = float(d_trib_ns)

In [ ]:
flexible_ns.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ns.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
l_aa__ab = 29 + 9 / 12
l_ac__ad = 29 + 9 / 12
l_stairwall = 20
wall_tributary = {
    "A-2": {"trib_width": float(t_w__A2), "trib_depth": t_l__A2, "wall_length": l_aa__ab},
    "A-2.4": {"trib_width": float(t_w__A24), "trib_depth": t_l__A24, "wall_length": l_ac__ad},
    "A-3.7": {"trib_width": float(t_w__A37), "trib_depth": t_l__A37, "wall_length": l_stairwall},
    "A-4": {"trib_width": float(t_w__A4), "trib_depth": t_l__A4, "wall_length": l_stairwall},
    "A-5": {"trib_width": float(t_w__A5), "trib_depth": t_l__A5, "wall_length": l_aa__ab},
    "A-5.1": {"trib_width": float(t_w__A51), "trib_depth": t_l__A51, "wall_length": l_ac__ad},
    "A-6": {"trib_width": float(t_w__A6), "trib_depth": t_l__A6, "wall_length": l_aa__ab},
    "A-6.1": {"trib_width": float(t_w__A61), "trib_depth": t_l__A61, "wall_length": l_ac__ad},
    "A-6.8": {"trib_width": float(t_w__A68), "trib_depth": t_l__A68, "wall_length": l_ac__ad},
    "A-7": {"trib_width": float(t_w__A7), "trib_depth": t_l__A7, "wall_length": l_aa__ab},
    "A-7.8": {"trib_width": float(t_w__A78), "trib_depth": t_l__A78, "wall_length": l_stairwall},
    "A-8": {"trib_width": float(t_w__A8), "trib_depth": t_l__A8, "wall_length": l_aa__ab},
    "A-8.2": {"trib_width": float(t_w__A82), "trib_depth": t_l__A82, "wall_length": l_stairwall},
    "A-8.9": {"trib_width": float(t_w__A89), "trib_depth": t_l__A89, "wall_length": l_ac__ad},
    "A-9": {"trib_width": float(t_w__A9), "trib_depth": t_l__A9, "wall_length": l_aa__ab},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("--").str[0]
# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line unit shear"] = flexible_ns["line shear demand"] / flexible_ns["line wall length"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line cumulative unit shear"] = (
    flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]
)

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [43]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [44]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [45]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [46]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible["line"] = flexible["line"].astype("string")
flexible.update(flexible_ns)

In [47]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [48]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [49]:
display_table(
    dataframe=flexible_ns,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Roof",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_level4_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,4.69743,1662.437500,7809.183325,7809.183325,30.75,253.957181,177.770027,310.714286,1,6.0
A-2.4,4.69743,2146.791667,10084.402984,10084.402984,30.75,327.948065,229.563645,310.714286,1,6.0
A-3.7,4.69743,741.187500,3481.676193,3481.676193,20.00,174.083810,121.858667,310.714286,1,6.0
A-4,4.69743,505.291667,2373.572094,2373.572094,20.00,118.678605,83.075023,310.714286,1,6.0
A-5,4.69743,770.500000,3619.369601,3619.369601,30.75,117.703076,82.392154,310.714286,1,6.0
A-5.1,4.69743,1384.666667,6504.374356,6504.374356,30.75,211.524369,148.067059,310.714286,1,6.0
A-6,4.69743,988.250000,4642.234923,4642.234923,30.75,150.966989,105.676893,310.714286,1,6.0
A-6.1,4.69743,826.333333,3881.642761,3881.642761,30.75,126.232285,88.362599,310.714286,1,6.0
A-6.8,4.69743,905.895833,4255.382013,4255.382013,30.75,138.386407,96.870485,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,5.229769,1662.437500,8694.164450,16503.347775,30.75,536.694237,375.685966,460.714286,1,4.0
A-2.4,5.229769,2146.791667,11227.224957,21311.627941,30.75,693.061071,485.142750,600.000000,1,3.0
A-3.7,5.229769,741.187500,3876.239566,7357.915758,20.00,367.895788,257.527052,310.714286,1,6.0
A-4,5.229769,505.291667,2642.558800,5016.130894,20.00,250.806545,175.564581,310.714286,1,6.0
A-5,5.229769,770.500000,4029.537176,7648.906777,30.75,248.744936,174.121455,310.714286,1,6.0
A-5.1,5.229769,1384.666667,7241.487099,13745.861455,30.75,447.019885,312.913919,460.714286,1,4.0
A-6,5.229769,988.250000,5168.319421,9810.554345,30.75,319.042418,223.329692,310.714286,1,6.0
A-6.1,5.229769,826.333333,4321.532623,8203.175384,30.75,266.769931,186.738952,310.714286,1,6.0
A-6.8,5.229769,905.895833,4737.626136,8993.008149,30.75,292.455550,204.718885,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,3.550605,1662.437500,5902.658971,22406.006745,30.75,728.650626,510.055438,600.000000,1,3.0
A-2.4,3.550605,2146.791667,7622.409317,28934.037258,30.75,940.944301,658.661011,769.642857,1,2.0
A-3.7,3.550605,741.187500,2631.664075,9989.579834,20.00,499.478992,349.635294,460.714286,1,4.0
A-4,3.550605,505.291667,1794.091140,6810.222033,20.00,340.511102,238.357771,310.714286,1,6.0
A-5,3.550605,770.500000,2735.741185,10384.647963,30.75,337.712129,236.398490,310.714286,1,6.0
A-5.1,3.550605,1384.666667,4916.404449,18662.265904,30.75,606.902956,424.832069,460.714286,1,4.0
A-6,3.550605,988.250000,3508.885433,13319.439778,30.75,433.152513,303.206759,310.714286,1,6.0
A-6.1,3.550605,826.333333,2933.983300,11137.158685,30.75,362.184022,253.528816,310.714286,1,6.0
A-6.8,3.550605,905.895833,3216.478314,12209.486463,30.75,397.056470,277.939529,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,1.866927,1662.437500,3103.649174,25509.655919,30.75,829.582306,580.707614,600.000000,1,3.0
A-2.4,1.866927,2146.791667,4007.902963,32941.940221,30.75,1071.282609,749.897826,769.642857,1,2.0
A-3.7,1.866927,741.187500,1383.742831,11373.322664,20.00,568.666133,398.066293,460.714286,1,4.0
A-4,1.866927,505.291667,943.342570,7753.564603,20.00,387.678230,271.374761,310.714286,1,6.0
A-5,1.866927,770.500000,1438.467123,11823.115086,30.75,384.491548,269.144083,310.714286,1,6.0
A-5.1,1.866927,1384.666667,2585.071352,21247.337256,30.75,690.970317,483.679222,600.000000,1,3.0
A-6,1.866927,988.250000,1844.990441,15164.430219,30.75,493.152202,345.206542,460.714286,1,4.0
A-6.1,1.866927,826.333333,1542.703871,12679.862556,30.75,412.353254,288.647278,310.714286,1,6.0
A-6.8,1.866927,905.895833,1691.241237,13900.727701,30.75,452.056185,316.439330,460.714286,1,4.0


#### East-West Direction

Most likely, the shear walls in the E-W direction will be governed by a rigid diaphragm assumption. 

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew} through \autoref{tab:shear_wall_forces_level2_ew}.

The walls in the E-W direction are named to be easily found on the plan because there are only 2 lines with walls in this direction. The naming goes _[E-W line]-[N-S line intersection of wall left edge]_. For example, wall "A-C--A-3.8" is the wall on line "A-C" whose left edge begins at "A-4.2".

In [50]:
# Plan dimensions
l_AA__AB = (30 + 9 / 12) * ft
l_AB__AC = (5 + 6 / 12) * ft
l_AC__AD = (30 + 9 / 12) * ft

d_trib_ew = L_floor

In [51]:
# Wall tributary widths and lengths
t_w__AB = l_AA__AB + l_AB__AC / 2
t_w__AC = l_AB__AC / 2 + l_AC__AD

# t_l__AB__1 = 33.75
# t_l__AB__2 = 22.25
# t_l__AB__3 = 16
# t_l__AB__4 = 19
# t_l__AB__5 = 16
# t_l__AB__6 = 34.5
# t_l__AB__7 = 28.25
# t_l__AB__8 = 28.5
# t_l__AB__9 = 31.75
t_l__AB__1 = 33
t_l__AB__2 = 21.5
t_l__AB__3 = 16.75
t_l__AB__4 = 19.5
t_l__AB__5 = 16
t_l__AB__6 = 34
t_l__AB__7 = 28.75
t_l__AB__8 = 28
t_l__AB__9 = 32.5


# t_l__AC__1 = 44
# t_l__AC__2 = 31
# t_l__AC__3 = 27.5
# t_l__AC__4 = 31
# t_l__AC__5 = 15.5
# t_l__AC__6 = 12.34
# # t_l__AC__6 = 12.34
# # t_l__AC__7 = 19
# # t_l__AC__8 = 21.33
# t_l__AC__8 = 21.33
# t_l__AC__9 = 28.33
t_l__AC__1 = 46.67
t_l__AC__2 = 14
t_l__AC__3 = 21.5
t_l__AC__4 = 20.33
t_l__AC__5 = 30.5
t_l__AC__6 = 16
t_l__AC__7 = 29
t_l__AC__8 = 23.5
t_l__AC__9 = 28.5

In [52]:
# Wall segment lengths
# l_AB__1 = 21
# l_AB__2 = 13.25
# l_AB__3 = 9
# l_AB__4 = 12
# l_AB__5 = 11
# l_AB__6 = 29.75
# l_AB__7 = 12.75
# l_AB__8 = 16.25
# l_AB__9 = 13.75
l_AB__1 = 20
l_AB__2 = 13
l_AB__3 = 9
l_AB__4 = 12
l_AB__5 = 10.67
l_AB__6 = 29
l_AB__7 = 13
l_AB__8 = 11.67
l_AB__9 = 10

# l_AC__1 = 17.25
# l_AC__2 = 23
# l_AC__3 = 22.5
# l_AC__4 = 26
# # l_AC__5 = 11.25
# l_AC__5 = 16
# l_AC__6 = 7
# l_AC__7 = 6.75
# l_AC__8 = 10.25
# l_AC__9 = 25.25
l_AC__1 = 16
l_AC__2 = 9.34
l_AC__3 = 16.75
l_AC__4 = 15
l_AC__5 = 26
l_AC__6 = 11.5
l_AC__7 = 16
l_AC__8 = 9.75
l_AC__9 = 24

In [ ]:
flexible_ew.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ew.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "A-B--A-1": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__1, "wall_length": l_AB__1},
    "A-B--A-2.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__2, "wall_length": l_AB__2},
    "A-B--A-3.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__3, "wall_length": l_AB__3},
    "A-B--A-4": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__4, "wall_length": l_AB__4},
    "A-B--A-4.8": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__5, "wall_length": l_AB__5},
    "A-B--A-5.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__6, "wall_length": l_AB__6},
    "A-B--A-6.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__7, "wall_length": l_AB__7},
    "A-B--A-7.9": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__8, "wall_length": l_AB__8},
    "A-B--A-8.8": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__9, "wall_length": l_AB__9},
    "A-C--A-1.8": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__1, "wall_length": l_AC__1},
    "A-C--A-3.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__2, "wall_length": l_AC__2},
    "A-C--A-3.8": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__3, "wall_length": l_AC__3},
    "A-C--A-4.6": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__7, "wall_length": l_AC__7},
    "A-C--A-5.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__4, "wall_length": l_AC__4},
    "A-C--A-6.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__5, "wall_length": l_AC__5},
    "A-C--A-6.9": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__6, "wall_length": l_AC__6},
    "A-C--A-8.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__8, "wall_length": l_AC__8},
    "A-C--A-9.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__9, "wall_length": l_AC__9},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("--").str[0]

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [54]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [55]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [56]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [57]:
# Update dataframe
flexible.update(flexible_ew)

In [58]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [59]:
display_table(
    dataframe=flexible_ew,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Roof",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 4",
    label="tab:shear_wall_forces_level4_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B--A-1,4.69743,1105.500,5640.283000,5640.283000,20.00,282.014150,197.409905,310.714286,1,6.0
A-B--A-2.5,4.69743,720.250,3666.183950,3666.183950,13.00,282.014150,197.409905,310.714286,1,6.0
A-B--A-3.5,4.69743,561.125,2538.127350,2538.127350,9.00,282.014150,197.409905,310.714286,1,6.0
A-B--A-4,4.69743,653.250,3384.169800,3384.169800,12.00,282.014150,197.409905,310.714286,1,6.0
A-B--A-4.8,4.69743,536.000,3009.090981,3009.090981,10.67,282.014150,197.409905,310.714286,1,6.0
A-B--A-5.5,4.69743,1139.000,8178.410351,8178.410351,29.00,282.014150,197.409905,310.714286,1,6.0
A-B--A-6.5,4.69743,963.125,3666.183950,3666.183950,13.00,282.014150,197.409905,310.714286,1,6.0
A-B--A-7.9,4.69743,938.000,3291.105131,3291.105131,11.67,282.014150,197.409905,310.714286,1,6.0
A-B--A-8.8,4.69743,1088.750,2820.141500,2820.141500,10.00,282.014150,197.409905,310.714286,1,6.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B--A-1,5.229769,1105.500,6279.471990,11919.754990,20.00,595.987750,417.191425,460.714286,1,4.0
A-B--A-2.5,5.229769,720.250,4081.656793,7747.840744,13.00,595.987750,417.191425,460.714286,1,4.0
A-B--A-3.5,5.229769,561.125,2825.762395,5363.889746,9.00,595.987750,417.191425,460.714286,1,4.0
A-B--A-4,5.229769,653.250,3767.683194,7151.852994,12.00,595.987750,417.191425,460.714286,1,4.0
A-B--A-4.8,5.229769,536.000,3350.098307,6359.189287,10.67,595.987750,417.191425,460.714286,1,4.0
A-B--A-5.5,5.229769,1139.000,9105.234385,17283.644736,29.00,595.987750,417.191425,460.714286,1,4.0
A-B--A-6.5,5.229769,963.125,4081.656793,7747.840744,13.00,595.987750,417.191425,460.714286,1,4.0
A-B--A-7.9,5.229769,938.000,3664.071906,6955.177037,11.67,595.987750,417.191425,460.714286,1,4.0
A-B--A-8.8,5.229769,1088.750,3139.735995,5959.877495,10.00,595.987750,417.191425,460.714286,1,4.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B--A-1,3.550605,1105.500,4263.271288,16183.026278,20.00,809.151314,566.405920,600.0,1,3.0
A-B--A-2.5,3.550605,720.250,2771.126337,10518.967081,13.00,809.151314,566.405920,600.0,1,3.0
A-B--A-3.5,3.550605,561.125,1918.472080,7282.361825,9.00,809.151314,566.405920,600.0,1,3.0
A-B--A-4,3.550605,653.250,2557.962773,9709.815767,12.00,809.151314,566.405920,600.0,1,3.0
A-B--A-4.8,3.550605,536.000,2274.455232,8633.644519,10.67,809.151314,566.405920,600.0,1,3.0
A-B--A-5.5,3.550605,1139.000,6181.743367,23465.388103,29.00,809.151314,566.405920,600.0,1,3.0
A-B--A-6.5,3.550605,963.125,2771.126337,10518.967081,13.00,809.151314,566.405920,600.0,1,3.0
A-B--A-7.9,3.550605,938.000,2487.618796,9442.795833,11.67,809.151314,566.405920,600.0,1,3.0
A-B--A-8.8,3.550605,1088.750,2131.635644,8091.513139,10.00,809.151314,566.405920,600.0,1,3.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B--A-1,1.866927,1105.500,2241.650496,18424.676774,20.00,921.233839,644.863687,769.642857,1,2.0
A-B--A-2.5,1.866927,720.250,1457.072822,11976.039903,13.00,921.233839,644.863687,769.642857,1,2.0
A-B--A-3.5,1.866927,561.125,1008.742723,8291.104548,9.00,921.233839,644.863687,769.642857,1,2.0
A-B--A-4,1.866927,653.250,1344.990298,11054.806064,12.00,921.233839,644.863687,769.642857,1,2.0
A-B--A-4.8,1.866927,536.000,1195.920540,9829.565059,10.67,921.233839,644.863687,769.642857,1,2.0
A-B--A-5.5,1.866927,1139.000,3250.393219,26715.781322,29.00,921.233839,644.863687,769.642857,1,2.0
A-B--A-6.5,1.866927,963.125,1457.072822,11976.039903,13.00,921.233839,644.863687,769.642857,1,2.0
A-B--A-7.9,1.866927,938.000,1308.003064,10750.798898,11.67,921.233839,644.863687,769.642857,1,2.0
A-B--A-8.8,1.866927,1088.750,1120.825248,9212.338387,10.00,921.233839,644.863687,769.642857,1,2.0


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection limits and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [60]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [61]:
%%render params
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

Each relative stiffness is relative to the level. The sum of all relative stiffnesses on each level is $1.0$.

\begin{align*}
k &= \frac{F_x}{\delta_{sw}} \\
R &= \frac{k_{wall}}{\sum {k_{wall}}}
\end{align*}

##### North-South Direction

Rigidities in the N-S direction can be found in \autoref{tab:wall_rigidities_ns_roof} through \autoref{tab:wall_rigidities_ns_level2}.

In [62]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"] * rigid_ns["wall height"] / (1000 * rigid_ns["shear stiffness"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

In [63]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]

In [64]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [65]:
display_table(
    dataframe=rigid_ns,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Roof",
    label="tab:wall_rigidities_ns_roof",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 3",
    label="tab:wall_rigidities_ns_level3",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 2",
    label="tab:wall_rigidities_ns_level2",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
A-2,7809.183325,0.266632,29288.294664,0.089216
A-2.4,10084.402984,0.320628,31452.035556,0.095807
A-3.7,3481.676193,0.253490,13734.941315,0.041838
A-4,2373.572094,0.212596,11164.701275,0.034009
A-5,3619.369601,0.167197,21647.313323,0.065941
A-5.1,6504.374356,0.235665,27600.052773,0.084073
A-6,4642.234923,0.191472,24244.951226,0.073853
A-6.1,3881.642761,0.173422,22382.701677,0.068181
A-6.8,4255.382013,0.182291,23343.859376,0.071108


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
A-2,16503.347775,0.426475,38697.132367,0.092065
A-2.4,21311.627941,0.480798,44325.563615,0.105455
A-3.7,7357.915758,0.417312,17631.691186,0.041948
A-4,5016.130894,0.326268,15374.276740,0.036577
A-5,7648.906777,0.276384,27674.949785,0.065842
A-5.1,13745.861455,0.369480,37203.260574,0.088511
A-6,9810.554345,0.330367,29695.919368,0.070650
A-6.1,8203.175384,0.290226,28264.819587,0.067245
A-6.8,8993.008149,0.309950,29014.353815,0.069028


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
A-2,22406.006745,0.501104,44713.322701,0.093657
A-2.4,28934.037258,0.531796,54408.199416,0.113964
A-3.7,9989.579834,0.453519,22026.835054,0.046137
A-4,6810.222033,0.396019,17196.719905,0.036020
A-5,10384.647963,0.344704,30126.271281,0.063103
A-5.1,18662.265904,0.471098,39614.442666,0.082977
A-6,13319.439778,0.417995,31865.047545,0.066745
A-6.1,11137.158685,0.363497,30638.955599,0.064176
A-6.8,12209.486463,0.390276,31284.222309,0.065528


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
A-2,25509.655919,0.615150,41469.006117,0.088346
A-2.4,32941.940221,0.654368,50341.599805,0.107248
A-3.7,11373.322664,0.549028,20715.375977,0.044132
A-4,7753.564603,0.476238,16280.850199,0.034685
A-5,11823.115086,0.418384,28259.014036,0.060203
A-5.1,21247.337256,0.527989,40242.038416,0.085732
A-6,15164.430219,0.438714,34565.667563,0.073639
A-6.1,12679.862556,0.441926,28692.246158,0.061126
A-6.8,13900.727701,0.409946,33908.720614,0.072239


##### East-West Direction

Rigidities in the E-W direction can be found in \autoref{tab:wall_rigidities_ew_roof}, \autoref{tab:wall_rigidities_ew_level4}, \autoref{tab:wall_rigidities_ew_level3}, \autoref{tab:wall_rigidities_ew_level2},

In [66]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"] * rigid_ew["wall height"] / (1000 * rigid_ew["shear stiffness"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

In [67]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]

In [68]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [69]:
display_table(
    dataframe=rigid_ew,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Roof",
    label="tab:wall_rigidities_ew_roof",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 4",
    label="tab:wall_rigidities_ew_level4",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 3",
    label="tab:wall_rigidities_ew_level3",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 2",
    label="tab:wall_rigidities_ew_level2",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
A-B--A-1,5640.283000,0.333153,16929.992829,0.082548
A-B--A-2.5,3666.183950,0.404077,9072.993381,0.044239
A-B--A-3.5,2538.127350,0.494138,5136.476394,0.025045
A-B--A-4,3384.169800,0.420963,8039.113655,0.039198
A-B--A-4.8,3009.090981,0.448326,6711.830239,0.032726
A-B--A-5.5,8178.410351,0.292276,27981.772622,0.136435
A-B--A-6.5,3666.183950,0.404077,9072.993381,0.044239
A-B--A-7.9,3291.105131,0.427171,7704.426562,0.037566
A-B--A-8.8,2820.141500,0.464868,6066.543540,0.029580


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
A-B--A-1,11919.754990,0.515787,23109.840212,0.080334
A-B--A-2.5,7747.840744,0.595305,13014.902351,0.045242
A-B--A-3.5,5363.889746,0.696281,7703.627945,0.026779
A-B--A-4,7151.852994,0.614238,11643.450708,0.040475
A-B--A-4.8,6359.189287,0.644918,9860.465004,0.034277
A-B--A-5.5,17283.644736,0.469956,36777.137405,0.127844
A-B--A-6.5,7747.840744,0.595305,13014.902351,0.045242
A-B--A-7.9,6955.177037,0.621198,11196.389718,0.038921
A-B--A-8.8,5959.877495,0.663464,8982.971715,0.031226


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
A-B--A-1,16183.026278,0.600715,26939.616342,0.079646
A-B--A-2.5,10518.967081,0.683397,15392.181375,0.045506
A-B--A-3.5,7282.361825,0.788390,9237.007202,0.027309
A-B--A-4,9709.815767,0.703083,13810.341546,0.040830
A-B--A-4.8,8633.644519,0.734983,11746.725841,0.034729
A-B--A-5.5,23465.388103,0.553061,42428.235817,0.125437
A-B--A-6.5,10518.967081,0.683397,15392.181375,0.045506
A-B--A-7.9,9442.795833,0.710320,13293.725022,0.039302
A-B--A-8.8,8091.513139,0.754267,10727.650514,0.031716


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
A-B--A-1,18424.676774,0.637726,28891.211408,0.085656
A-B--A-2.5,11976.039903,0.733092,16336.330003,0.048434
A-B--A-3.5,8291.104548,0.854193,9706.363597,0.028777
A-B--A-4,11054.806064,0.755799,14626.653488,0.043365
A-B--A-4.8,9829.565059,0.792593,12401.785873,0.036769
A-B--A-5.5,26715.781322,0.582761,45843.452206,0.135916
A-B--A-6.5,11976.039903,0.733092,16336.330003,0.048434
A-B--A-7.9,10750.798898,0.764146,14069.041976,0.041712
A-B--A-8.8,9212.338387,0.814835,11305.770379,0.033519


#### Center of Rigidity and Center of Mass

We can find the center of rigidity by taking the weighted sum of all wall stiffnesses and their locations relative to a datum. In this case, we will use the north-west-most corner of the building (i.e., the intersection of lines A-1 and A-A).

$$
\bar{x}_r = \frac{\sum k_{ns} x}{\sum k_{ns}},\, \bar{y}_r = \frac{\sum k_{ew} y}{\sum k_{ew}}
$$

Note: each wall is assumed to have zero rigidity in the direction perpendicular to them.

The building is symmetric in each direction when projected to that direction, so the centers of mass are just half of each total plan dimension.

In [70]:
%%render
X_plan = L_floor + L_jog
Y_plan = w_building
CM_x = X_plan / 2
CM_y = Y_plan / 2

<IPython.core.display.Latex object>

An accidental torsion eccentricity of 5% must be added/subtracted.

In [71]:
%%render
CM_x__a__plus = CM_x + X_plan * 0.05
CM_x__a__minus = CM_x - X_plan * 0.05
CM_y__a__plus = CM_y + Y_plan * 0.05
CM_y__a__minus = CM_y - Y_plan * 0.05

<IPython.core.display.Latex object>

The eccentricity used is determined by the location of the wall in relation to the center of rigidity. Since the wall rotates about the center of rigidity, we want to position the center of mass to ensure each wall is getting the maximum possible force to be conservative. If the center of mass is positioned on the opposite side of the center of rigidity relative to the wall in question, that mass will _decrease_ the shear experienced by that wall.

This means we always want the center of mass to be on the same side of the center of rigidity as the wall. The eccentricities shown in the following tables are based on the worst-case scenario. This is not immediately obvious due to all torques being given a positive value. 

How the center of rigidity, center of mass, and eccentricity interact are illustrated in \autoref{fig:rigid_diaphragm_moments}. The moments generated by the rotation of the center of mass around the center of rigidity differ based on the chosen eccentricity.

![Moments applied by a rigid diaphragm](images/rigid_diaphragm_moments.png){width=50% #fig:rigid_diaphragm_moments}

$$
d_x = \left| x - \text{CR}_x \right|\text{ and }\, d_y = \left| y - \text{CR}_y \right|
$$

In [72]:
# Update dataframe
rigid = rigid.reindex(columns=rigid_ns.columns)
rigid.update(rigid_ns)
rigid.update(rigid_ew)

In [73]:
# Location of wall lines of rigidity
wall_rigidity_loc = {
    # N-S
    "A-2": {"x": float(l_A1__A2), "y": 0},
    "A-2.4": {"x": float(l_A1__A24), "y": 0},
    "A-3.7": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A37), "y": 0},
    "A-4": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4), "y": 0},
    "A-5": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5), "y": 0},
    "A-5.1": {"x": float(l_A1__A24 + l_A24__A35 + l_A35__A51), "y": 0},
    "A-6": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5 + l_A5__A6), "y": 0},
    "A-6.1": {"x": float(l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61), "y": 0},
    "A-6.8": {"x": float(l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61 + l_A61__A68), "y": 0},
    "A-7": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5 + l_A5__A6 + l_A6__A7), "y": 0},
    "A-7.8": {"x": float(l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61 + l_A61__A68 + l_A68__A78), "y": 0},
    "A-8": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5 + l_A5__A6 + l_A6__A7 + l_A7__A8), "y": 0},
    "A-8.2": {
        "x": float(l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61 + l_A61__A68 + l_A68__A78 + l_A78__A82),
        "y": 0,
    },
    "A-8.9": {
        "x": float(
            l_A1__A24 + l_A24__A35 + l_A35__A51 + l_A51__A61 + l_A61__A68 + l_A68__A78 + l_A78__A82 + l_A82__A89
        ),
        "y": 0,
    },
    "A-9": {"x": float(l_A1__A2 + l_A2__A3 + l_A3__A4 + l_A4__A5 + l_A5__A6 + l_A6__A7 + l_A7__A8 + l_A8__A9), "y": 0},
    # E-W
    "A-B--A-1": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-2.5": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-3.5": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-4": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-4.8": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-5.5": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-6.5": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-7.9": {"x": 0, "y": float(l_AA__AB)},
    "A-B--A-8.8": {"x": 0, "y": float(l_AA__AB)},
    "A-C--A-1.8": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-3.2": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-3.8": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-4.6": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-5.5": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-6.5": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-6.9": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-8.5": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
    "A-C--A-9.2": {"x": 0, "y": float(l_AA__AB + l_AB__AC)},
}

In [74]:
# Calc centers of rigidity
rigid["x"] = rigid.index.get_level_values("Wall").map({key: value["x"] for key, value in wall_rigidity_loc.items()})
rigid["y"] = rigid.index.get_level_values("Wall").map({key: value["y"] for key, value in wall_rigidity_loc.items()})

rigid["kx"] = rigid["relative wall stiffness"] * rigid["x"]
rigid["CRx"] = rigid.groupby(["Level", "Direction"])["kx"].transform("sum")  # all relative wall rigidities sum to 1
rigid["ky"] = rigid["relative wall stiffness"] * rigid["y"]
rigid["CRy"] = rigid.groupby(["Level", "Direction"])["ky"].transform("sum")  # all relative wall rigidities sum to 1

# x eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_x = float(X_plan)
CMx = plan_dim_x / 2
CMxa_plus = CMx + plan_dim_x * 0.05
CMxa_minus = CMx - plan_dim_x * 0.05
rigid["ex"] = np.where(rigid["x"] <= rigid["CRx"], rigid["CRx"] - CMxa_minus, CMxa_plus - rigid["CRx"])
rigid["ex"] = np.where(rigid["ex"] <= 0, 0, rigid["ex"])  # assign 0s to negatives

# y eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_y = float(w_building)
CMy = plan_dim_y / 2
CMya_plus = CMy + plan_dim_y * 0.05
CMya_minus = CMy - plan_dim_y * 0.05
rigid["ey"] = np.where(rigid["y"] <= rigid["CRy"], rigid["CRy"] - CMya_minus, CMya_plus - rigid["CRy"])
rigid["ey"] = np.where(rigid["ey"] <= 0, 0, rigid["ey"])  # assign 0s to negatives

# distance from CR
rigid["dx"] = abs(rigid["x"] - rigid["CRx"])
rigid["dy"] = abs(rigid["y"] - rigid["CRy"])

# Torque
rigid["torque x"] = rigid["total level cumulative shear"] * rigid["ex"]
rigid["torque y"] = rigid["total level cumulative shear"] * rigid["ey"]

# Polar moment of inertia
rigid["Jx wall"] = rigid["relative wall stiffness"] * rigid["dx"] ** 2
rigid["Jy wall"] = rigid["relative wall stiffness"] * rigid["dy"] ** 2
rigid["J"] = (rigid["Jx wall"] + rigid["Jy wall"]).groupby(["Level"]).transform("sum")

rigid["torsional shear force x"] = rigid["torque x"] * rigid["relative wall stiffness"] * rigid["dx"] / rigid["J"]
rigid["torsional shear force y"] = rigid["torque y"] * rigid["relative wall stiffness"] * rigid["dy"] / rigid["J"]
rigid["torsional shear force"] = np.where(
    rigid["torsional shear force x"] == 0, rigid["torsional shear force y"], rigid["torsional shear force x"]
)
rigid["shear force demand"] = rigid["torsional shear force"] + rigid["direct shear force"]

The centers of rigidity for each floor are given in \autoref{tab:center_of_rigidity_roof} through \autoref{tab:center_of_rigidity_level2}.

In [75]:
column_name_map = {
    "x": "$x$ [ft]",
    "y": "$y$ [ft]",
    "CRx": "CR$_x$ [ft]",
    "dx": "$d_x$ [ft]",
    "ex": "e$_x$ [ft]",
    "CRy": "CR$_y$ [ft]",
    "dy": "$d_y$ [ft]",
    "ey": "e$_y$ [ft]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Roof",
    label="tab:center_of_rigidity_roof",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 4",
    label="tab:center_of_rigidity_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 3",
    label="tab:center_of_rigidity_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 2",
    label="tab:center_of_rigidity_level2",
)

,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
A-2,31.916667,0.00,124.690479,92.773812,18.940479,0.000000,0.000000,0.000000
A-2.4,40.083333,0.00,124.690479,84.607145,18.940479,0.000000,0.000000,0.000000
A-3.7,67.333333,0.00,124.690479,57.357145,18.940479,0.000000,0.000000,0.000000
A-4,76.166667,0.00,124.690479,48.523812,18.940479,0.000000,0.000000,0.000000
A-5,97.500000,0.00,124.690479,27.190479,18.940479,0.000000,0.000000,0.000000
A-5.1,98.083333,0.00,124.690479,26.607145,18.940479,0.000000,0.000000,0.000000
A-6,122.166667,0.00,124.690479,2.523812,18.940479,0.000000,0.000000,0.000000
A-6.1,122.750000,0.00,124.690479,1.940479,18.940479,0.000000,0.000000,0.000000
A-6.8,147.416667,0.00,124.690479,22.726188,4.559521,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
A-2,31.916667,0.00,122.363589,90.446922,16.613589,0.000000,0.000000,0.000000
A-2.4,40.083333,0.00,122.363589,82.280256,16.613589,0.000000,0.000000,0.000000
A-3.7,67.333333,0.00,122.363589,55.030256,16.613589,0.000000,0.000000,0.000000
A-4,76.166667,0.00,122.363589,46.196922,16.613589,0.000000,0.000000,0.000000
A-5,97.500000,0.00,122.363589,24.863589,16.613589,0.000000,0.000000,0.000000
A-5.1,98.083333,0.00,122.363589,24.280256,16.613589,0.000000,0.000000,0.000000
A-6,122.166667,0.00,122.363589,0.196922,16.613589,0.000000,0.000000,0.000000
A-6.1,122.750000,0.00,122.363589,0.386411,6.886411,0.000000,0.000000,0.000000
A-6.8,147.416667,0.00,122.363589,25.053078,6.886411,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
A-2,31.916667,0.00,122.125402,90.208736,16.375402,0.000000,0.000000,0.000000
A-2.4,40.083333,0.00,122.125402,82.042069,16.375402,0.000000,0.000000,0.000000
A-3.7,67.333333,0.00,122.125402,54.792069,16.375402,0.000000,0.000000,0.000000
A-4,76.166667,0.00,122.125402,45.958736,16.375402,0.000000,0.000000,0.000000
A-5,97.500000,0.00,122.125402,24.625402,16.375402,0.000000,0.000000,0.000000
A-5.1,98.083333,0.00,122.125402,24.042069,16.375402,0.000000,0.000000,0.000000
A-6,122.166667,0.00,122.125402,0.041264,7.124598,0.000000,0.000000,0.000000
A-6.1,122.750000,0.00,122.125402,0.624598,7.124598,0.000000,0.000000,0.000000
A-6.8,147.416667,0.00,122.125402,25.291264,7.124598,0.000000,0.000000,0.000000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
A-2,31.916667,0.00,123.560515,91.643849,17.810515,0.000000,0.000000,0.000000
A-2.4,40.083333,0.00,123.560515,83.477182,17.810515,0.000000,0.000000,0.000000
A-3.7,67.333333,0.00,123.560515,56.227182,17.810515,0.000000,0.000000,0.000000
A-4,76.166667,0.00,123.560515,47.393849,17.810515,0.000000,0.000000,0.000000
A-5,97.500000,0.00,123.560515,26.060515,17.810515,0.000000,0.000000,0.000000
A-5.1,98.083333,0.00,123.560515,25.477182,17.810515,0.000000,0.000000,0.000000
A-6,122.166667,0.00,123.560515,1.393849,17.810515,0.000000,0.000000,0.000000
A-6.1,122.750000,0.00,123.560515,0.810515,17.810515,0.000000,0.000000,0.000000
A-6.8,147.416667,0.00,123.560515,23.856151,5.689485,0.000000,0.000000,0.000000


And now, we can get the forces in the shear walls using the rigid diaphragm assumption in \autoref{tab:rigid_forces_roof} through \autoref{tab:rigid_forces_level2}. These forces can be compared to the flexible shear wall forces obtained earlier, $F_{flex}$.

In [76]:
rigid["flexible shear force demand"] = flexible["cumulative shear demand"]

In [77]:
column_name_map = {
    "torque x": "$T_x$ [lbf-ft]",
    "torque y": "$T_y$ [lbf-ft]",
    "dx": "$d_x$ [ft]",
    "dy": "$d_y$ [ft]",
    "relative wall stiffness": "$R$ [-]",
    "torsional shear force": "$F_t$ [lbf]",
    "direct shear force": "$F_v$ [lbf]",
    "shear force demand": "$F_{tot}$ [lbf]",
    "flexible shear force demand": "$F_{flex}$ [lbf]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Roof",
    label="tab:rigid_forces_roof",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 4",
    label="tab:rigid_forces_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 3",
    label="tab:rigid_forces_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 2",
    label="tab:rigid_forces_level2",
)

,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
A-2,1.371052e+06,0.000000,92.773812,0.000000,0.089216,3364.783514,6458.116002,9822.899516,7809.183325
A-2.4,1.371052e+06,0.000000,84.607145,0.000000,0.095807,3295.288482,6935.224343,10230.512825,10084.402984
A-3.7,1.371052e+06,0.000000,57.357145,0.000000,0.041838,975.555488,3028.576614,4004.132103,3481.676193
A-4,1.371052e+06,0.000000,48.523812,0.000000,0.034009,670.871988,2461.834559,3132.706547,2373.572094
A-5,1.371052e+06,0.000000,27.190479,0.000000,0.065941,728.884131,4773.267347,5502.151478,3619.369601
A-5.1,1.371052e+06,0.000000,26.607145,0.000000,0.084073,909.380910,6085.855954,6995.236864,6504.374356
A-6,1.371052e+06,0.000000,2.523812,0.000000,0.073853,75.773253,5346.050676,5421.823929,4642.234923
A-6.1,1.371052e+06,0.000000,1.940479,0.000000,0.068181,53.784729,4935.421660,4989.206390,3881.642761
A-6.8,3.300519e+05,0.000000,22.726188,0.000000,0.071108,158.148551,5147.358476,5305.507026,4255.382013


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
A-2,2.541516e+06,0.000000,90.446922,0.000000,0.092065,6235.788950,14083.878359,20319.667309,16503.347775
A-2.4,2.541516e+06,0.000000,82.280256,0.000000,0.105455,6497.835768,16132.354207,22630.189975,21311.627941
A-3.7,2.541516e+06,0.000000,55.030256,0.000000,0.041948,1728.678904,6417.079994,8145.758898,7357.915758
A-4,2.541516e+06,0.000000,46.196922,0.000000,0.036577,1265.396324,5595.490679,6860.887003,5016.130894
A-5,2.541516e+06,0.000000,24.863589,0.000000,0.065842,1225.940773,10072.338763,11298.279535,7648.906777
A-5.1,2.541516e+06,0.000000,24.280256,0.000000,0.088511,1609.359577,13540.181518,15149.541095,13745.861455
A-6,2.541516e+06,0.000000,0.196922,0.000000,0.070650,10.418620,10807.873621,10818.292242,9810.554345
A-6.1,1.053470e+06,0.000000,0.386411,0.000000,0.067245,8.065740,10287.022747,10295.088487,8203.175384
A-6.8,1.053470e+06,0.000000,25.053078,0.000000,0.069028,536.812122,10559.816834,11096.628956,8993.008149


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
A-2,3.401056e+06,0.000000,90.208736,0.000000,0.093657,8122.458768,19451.839142,27574.297911,22406.006745
A-2.4,3.401056e+06,0.000000,82.042069,0.000000,0.113964,8988.825162,23669.445237,32658.270400,28934.037258
A-3.7,3.401056e+06,0.000000,54.792069,0.000000,0.046137,2430.366526,9582.433745,12012.800271,9989.579834
A-4,3.401056e+06,0.000000,45.958736,0.000000,0.036020,1591.533064,7481.166891,9072.699955,6810.222033
A-5,3.401056e+06,0.000000,24.625402,0.000000,0.063103,1493.931524,13105.968144,14599.899668,10384.647963
A-5.1,3.401056e+06,0.000000,24.042069,0.000000,0.082977,1917.906203,17233.650284,19151.556486,18662.265904
A-6,1.479729e+06,0.000000,0.041264,0.000000,0.066745,1.152022,13862.395851,13863.547873,13319.439778
A-6.1,1.479729e+06,0.000000,0.624598,0.000000,0.064176,16.766589,13329.003523,13345.770112,11137.158685
A-6.8,1.479729e+06,0.000000,25.291264,0.000000,0.065528,693.212292,13609.716820,14302.929112,12209.486463


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
A-2,4.211515e+06,0.000000,91.643849,0.000000,0.088346,9924.348008,20890.475981,30814.823989,25509.655919
A-2.4,4.211515e+06,0.000000,83.477182,0.000000,0.107248,10974.123194,25360.144359,36334.267553,32941.940221
A-3.7,4.211515e+06,0.000000,56.227182,0.000000,0.044132,3041.684581,10435.602509,13477.287090,11373.322664
A-4,4.211515e+06,0.000000,47.393849,0.000000,0.034685,2014.995571,8201.660514,10216.656085,7753.564603
A-5,4.211515e+06,0.000000,26.060515,0.000000,0.060203,1923.158335,14235.794615,16158.952950,11823.115086
A-5.1,4.211515e+06,0.000000,25.477182,0.000000,0.085732,2677.357701,20272.377268,22949.734969,21247.337256
A-6,4.211515e+06,0.000000,1.393849,0.000000,0.073639,125.815912,17412.841917,17538.657829,15164.430219
A-6.1,4.211515e+06,0.000000,0.810515,0.000000,0.061126,60.729646,14454.040145,14514.769791,12679.862556
A-6.8,1.345349e+06,0.000000,23.856151,0.000000,0.072239,674.812760,17081.897539,17756.710299,13900.727701


### Final Shear Wall Nailing Selection

With the envelope procedure complete, we can now select the shear wall sheathing and nailing specifications in \autoref{tab:shear_walls_roof} through \autoref{tab:shear_walls_level2}.

In [78]:
shear_walls["wall length"] = rigid["wall length"]
shear_walls["rigid shear force demand"] = rigid["shear force demand"]
shear_walls["flexible shear force demand"] = rigid["flexible shear force demand"]
shear_walls["shear force demand"] = np.where(
    shear_walls["flexible shear force demand"] >= shear_walls["rigid shear force demand"],
    shear_walls["flexible shear force demand"],
    shear_walls["rigid shear force demand"],
)
shear_walls["unit shear demand"] = shear_walls["shear force demand"] / shear_walls["wall length"]
shear_walls["adjusted unit shear demand"] = (
    shear_walls["shear force demand"] / shear_walls["wall length"] * LOAD_FACTOR_SEISMIC_ASD
)
shear_walls["sheathed sides"] = 1

# using sheathing from the NS calc in memory

shear_walls[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        shear_walls[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (shear_walls["shear dcr"] > shear_dcr_check_value).any():
    shear_walls.loc[shear_walls["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    shear_walls.loc[
        shear_walls["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        shear_walls.loc[
            shear_walls["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

column_name_map = {
    "shear force demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    caption="Final shear wall design values - Roof",
    label="tab:shear_walls_roof",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    caption="Final shear wall design values - Level 4",
    label="tab:shear_walls_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    caption="Final shear wall design values - Level 3",
    label="tab:shear_walls_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    caption="Final shear wall design values - Level 2",
    label="tab:shear_walls_level2",
)

,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
A-2,9822.899516,30.75,319.443887,223.610721,310.714286,1,6.0
A-2.4,10230.512825,30.75,332.699604,232.889723,310.714286,1,6.0
A-3.7,4004.132103,20.00,200.206605,140.144624,310.714286,1,6.0
A-4,3132.706547,20.00,156.635327,109.644729,310.714286,1,6.0
A-5,5502.151478,30.75,178.931755,125.252229,310.714286,1,6.0
A-5.1,6995.236864,30.75,227.487378,159.241164,310.714286,1,6.0
A-6,5421.823929,30.75,176.319477,123.423634,310.714286,1,6.0
A-6.1,4989.206390,30.75,162.250614,113.575430,310.714286,1,6.0
A-6.8,5305.507026,30.75,172.536814,120.775770,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
A-2,20319.667309,30.75,660.802189,462.561532,600.000000,1,3.0
A-2.4,22630.189975,30.75,735.941137,515.158796,600.000000,1,3.0
A-3.7,8145.758898,20.00,407.287945,285.101561,310.714286,1,6.0
A-4,6860.887003,20.00,343.044350,240.131045,310.714286,1,6.0
A-5,11298.279535,30.75,367.423725,257.196607,310.714286,1,6.0
A-5.1,15149.541095,30.75,492.668003,344.867602,460.714286,1,4.0
A-6,10818.292242,30.75,351.814382,246.270067,310.714286,1,6.0
A-6.1,10295.088487,30.75,334.799626,234.359738,310.714286,1,6.0
A-6.8,11096.628956,30.75,360.865982,252.606188,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
A-2,27574.297911,30.75,896.725135,627.707595,769.642857,1,2.0
A-2.4,32658.270400,30.75,1062.057574,743.440302,769.642857,1,2.0
A-3.7,12012.800271,20.00,600.640014,420.448009,460.714286,1,4.0
A-4,9072.699955,20.00,453.634998,317.544498,460.714286,1,4.0
A-5,14599.899668,30.75,474.793485,332.355440,460.714286,1,4.0
A-5.1,19151.556486,30.75,622.814845,435.970392,460.714286,1,4.0
A-6,13863.547873,30.75,450.847085,315.592960,460.714286,1,4.0
A-6.1,13345.770112,30.75,434.008784,303.806149,310.714286,1,6.0
A-6.8,14302.929112,30.75,465.135906,325.595134,460.714286,1,4.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
A-2,30814.823989,30.75,1002.108097,701.475668,769.642857,1,2.0
A-2.4,36334.267553,30.75,1181.602197,827.121538,921.428571,2,4.0
A-3.7,13477.287090,20.00,673.864354,471.705048,600.000000,1,3.0
A-4,10216.656085,20.00,510.832804,357.582963,460.714286,1,4.0
A-5,16158.952950,30.75,525.494405,367.846083,460.714286,1,4.0
A-5.1,22949.734969,30.75,746.332845,522.432991,600.000000,1,3.0
A-6,17538.657829,30.75,570.362856,399.253999,460.714286,1,4.0
A-6.1,14514.769791,30.75,472.025034,330.417524,460.714286,1,4.0
A-6.8,17756.710299,30.75,577.453993,404.217795,460.714286,1,4.0


### Shear Wall Holddown Forces

## Diaphragm Design

We must first get the diaphragm inertial design forces. These are distinct from the seismic forces determined in \autoref{tab:seismic_forces_calculation_table}.

\begin{align*}
F_{px} &= \frac{\sum_{i=x}^n F_i}{\sum_{i=x}^n w_i} w_{px} \quad \text{(ASCE Eq. 12.10-1)} \\
F_{px} &\ge 0.2 S_{DS} I_e w_{px} \quad \text{(ASCE Eq. 12.10-2)} \\
F_{px} &\le 0.4 S_{DS} I_e w_{px}  \quad \text{(ASCE Eq. 12.10-3)}
\end{align*}

These forces are shown in \autoref{tab:diaphragm_forces}

In [79]:
seismic_loads["cumulative floor weight"] = seismic_loads.iloc[::-1]["floor weight"].cumsum()
seismic_loads["F_x,cum"] = seismic_loads.iloc[::-1]["F_x"].cumsum()
seismic_loads["F_px lower bound"] = 0.2 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px upper bound"] = 0.4 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px initial"] = (
    seismic_loads["F_x,cum"] / seismic_loads["cumulative floor weight"] * seismic_loads["floor weight"]
)

# Assign F_px with limits from ASCE
seismic_loads["F_px"] = np.where(
    seismic_loads["F_px initial"] <= seismic_loads["F_px lower bound"],
    seismic_loads["F_px lower bound"],
    np.where(
        seismic_loads["F_px initial"] >= seismic_loads["F_px upper bound"],
        seismic_loads["F_px upper bound"],
        seismic_loads["F_px initial"],
    ),
)

In [80]:
column_name_map = {
    "floor weight": "$w_{px}$ [kip]",
    "cumulative floor weight": r"$\sum w_x$ [kip]",
    "F_x,cum": r"$\sum F_x$ [kip]",
    "F_px initial": "$F_{px,init}$ [kip]",
    "F_px lower bound": "$F_{px,lower}$ [kip]",
    "F_px upper bound": "$F_{px,upper}$ [kip]",
    "F_px": "$F_{px}$ [kip]",
}
display_table(
    dataframe=seismic_loads.iloc[:0:-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Forces",
    label="tab:diaphragm_forces",
)

,$w_{px}$ [kip],$\sum w_x$ [kip],$\sum F_x$ [kip],"$F_{px,init}$ [kip]","$F_{px,lower}$ [kip]","$F_{px,upper}$ [kip]",$F_{px}$ [kip]
Level,,,,,,,
Roof,421.490,421.490,72.387392,72.387392,57.086178,114.172357,72.387392
Level 4,613.642,1035.132,152.978136,90.687766,83.111051,166.222101,90.687766
Level 3,615.454,1650.586,207.692959,77.442473,83.356466,166.712932,83.356466
Level 2,619.078,2269.664,236.462302,64.497921,83.847297,167.694594,83.847297


With these diaphragm forces, we can design the diaphragm for the worst-case cantilever and simply-supported conditions.

The longest simply-supported span in the N-S direction is about 34.5 feet, while the longest cantilever span is about 32 feet. The depth of the diaphragm in the N-S direction is 67 feet. The longest simply-supported span in the E-W direction is about 5.5 feet, while the longest cantilever span is about 31 feet. The depth of the diaphragm in the E-W direction is 230 feet.

Diaphragm unit shear demands are shown in \autoref{tab:diaphragm_unit_shears}

In [81]:
# Set up diaphragm dataframe
levels = ["Roof", "Level 4", "Level 3", "Level 2"]
directions = ["N-S", "E-W"]
index = pd.MultiIndex.from_product(
    [levels, directions],
    names=["Level", "Direction"],
)
diaphragms = pd.DataFrame(index=index)
diaphragms_ns = diaphragms.xs("N-S", level="Direction").copy()
diaphragms_ew = diaphragms.xs("E-W", level="Direction").copy()

In [82]:
diaphragms_ns["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ns["area load"] = diaphragms_ns["F_px"] / float(A_level)
diaphragms_ns["L_simp"] = 34.5
diaphragms_ns["L_cant"] = 32
diaphragms_ns["depth"] = 67
diaphragms_ns["V_simp"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_simp"] / 2
diaphragms_ns["V_cant"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_cant"]
diaphragms_ns["v_simp"] = diaphragms_ns["V_simp"] / diaphragms_ns["depth"]
diaphragms_ns["v_cant"] = diaphragms_ns["V_cant"] / diaphragms_ns["depth"]

We will assume the diaphragm sheathing has the following properties:

- Unblocked
- WSP Sheathing
- 15/32" thick
- 10d common nails
- Plywood
- Nailed face is 2"
- Edge adjoining cases 2 through 6

The diaphragm capacity checks for this chosen sheathing are shown in \autoref{tab:diaphragm_capacities}

In [83]:
# Set baseline sheathing properties. We set the number of sheathed sides later
from structural_tools.seismic.timber.sheathing import SheathingApplication

diaphragm_sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
    minimum_nominal_width_of_nailed_face=2,
    adjoining_panel_edge_location=(2, 3, 4, 5, 6),
)

diaphragms_ns[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ns["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ns[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ns["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [84]:
diaphragms_ew["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ew["area load"] = diaphragms_ew["F_px"] / float(A_level)
diaphragms_ew["L_simp"] = 5.5
diaphragms_ew["L_cant"] = 31
diaphragms_ew["depth"] = float(L_floor)
diaphragms_ew["V_simp"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_simp"] / 2
diaphragms_ew["V_cant"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_cant"]
diaphragms_ew["v_simp"] = diaphragms_ew["V_simp"] / diaphragms_ew["depth"]
diaphragms_ew["v_cant"] = diaphragms_ew["V_cant"] / diaphragms_ew["depth"]


In [85]:
diaphragms_ew[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ew["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ew[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ew["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [86]:
idx = pd.IndexSlice
diaphragms = diaphragms.reindex(columns=diaphragms_ns.columns)
diaphragms_ns["Direction"] = "N-S"
diaphragms_ew["Direction"] = "E-W"
diaphragms_ns = diaphragms_ns.set_index("Direction", append=True)
diaphragms_ew = diaphragms_ew.set_index("Direction", append=True)
diaphragms.update(diaphragms_ns)
diaphragms.update(diaphragms_ew)

In [87]:
column_name_map = {
    "F_px": "$F_{px}$ [lbf]",
    "area load": "$w$ [psf]",
    "V_simp": "$V_{simp}$ [lbf]",
    "V_cant": "$V_{cant}$ [lbf]",
    "v_simp": "$v_{simp}$ [plf]",
    "v_cant": "$v_{cant}$ [plf]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Unit Shear Demands",
    label="tab:diaphragm_unit_shears",
)

$F_{px}$ [lbf]  $w$ [psf]  $V_{simp}$ [lbf]  \
Level   Direction                                                
Roof    N-S          72387.392027   4.697430       5429.054402   
        E-W          72387.392027   4.697430       2971.124300   
Level 4 N-S          90687.766441   5.884995       6801.582483   
        E-W          90687.766441   5.884995       3722.259070   
Level 3 N-S          83356.466100   5.409245       6251.734957   
        E-W          83356.466100   5.409245       3421.347489   
Level 2 N-S          83847.296988   5.441096       6288.547274   
        E-W          83847.296988   5.441096       3441.493533   

                   $V_{cant}$ [lbf]  $v_{simp}$ [plf]  $v_{cant}$ [plf]  
Level   Direction                                                        
Roof    N-S            10071.289326         81.030663        150.317751  
        E-W            33492.673923         12.917932        145.620321  
Level 4 N-S            12617.428374        101.516156        188.319826  
        E-W            41960.011338         16.183735        182.434832  
Level 3 N-S            11597.421370         93.309477        173.095841  
        E-W            38567.917151         14.875424        167.686596  
Level 2 N-S            11665.710885         93.858915        174.115088  
        E-W            38795.018009         14.963015        168.673991

In [88]:
column_name_map = {
    "v_simp": "$v_{simp}$ [plf]",
    "adjusted unit shear capacity simply supported": "$v_{cap,simp}$ [plf]",
    "shear dcr simply supported": "DCR$_{simp}$ [-]",
    "v_cant": "$v_{cant}$ [plf]",
    "adjusted unit shear capacity cantilever": "$v_{cap,cant}$ [plf]",
    "shear dcr cantilever": "DCR$_{cant}$ [-]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Capacity Checks",
    label="tab:diaphragm_capacities",
)

$v_{simp}$ [plf]  $v_{cap,simp}$ [plf]  DCR$_{simp}$ [-]  \
Level   Direction                                                             
Roof    N-S               81.030663            189.285714          0.428087   
        E-W               12.917932            189.285714          0.068246   
Level 4 N-S              101.516156            189.285714          0.536312   
        E-W               16.183735            189.285714          0.085499   
Level 3 N-S               93.309477            189.285714          0.492956   
        E-W               14.875424            189.285714          0.078587   
Level 2 N-S               93.858915            189.285714          0.495858   
        E-W               14.963015            189.285714          0.079050   

                   $v_{cant}$ [plf]  $v_{cap,cant}$ [plf]  DCR$_{cant}$ [-]  
Level   Direction                                                            
Roof    N-S              150.317751            189.285714          0.794132  
        E-W              145.620321            189.285714          0.769315  
Level 4 N-S              188.319826            189.285714          0.994897  
        E-W              182.434832            189.285714          0.963807  
Level 3 N-S              173.095841            189.285714          0.914469  
        E-W              167.686596            189.285714          0.885891  
Level 2 N-S              174.115088            189.285714          0.919853  
        E-W              168.673991            189.285714          0.891108

In [89]:
if (
    (diaphragms[["shear dcr simply supported", "shear dcr cantilever"]] <= 1).all().all()
):  # double .all() because 2 columns
    display_text(r"$\therefore$ All shear capacities are OK")
else:
    display_text("You must check for new diaphragm nailing.")

'$\\therefore$ All shear capacities are OK'